In [8]:
# ==============================
# Email Spam Detection - Decision Tree
# ==============================

# 1️⃣ Import Libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import string
import re
from nltk.corpus import stopwords
import nltk
nltk.download('stopwords')

from sklearn.feature_extraction.text import CountVectorizer
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix

# ==============================
# 2️⃣ Load Dataset
# ==============================
df = pd.read_csv("EmailData.csv")

print("First 5 rows:")
print(df.head())
print("\nClass distribution:")
print(df['Category'].value_counts())
print("\nPercentage distribution:")
print(df['Category'].value_counts(normalize=True) * 100)

# ==============================
# 3️⃣ Data Cleaning
# ==============================
stop_words = set(stopwords.words('english'))

def clean_text(text):
    # Lowercase
    text = text.lower()
    # Remove punctuation
    text = text.translate(str.maketrans('', '', string.punctuation))
    # Remove numbers
    text = re.sub(r'\d+', '', text)
    # Remove stopwords
    words = text.split()
    words = [word for word in words if word not in stop_words]
    # Join back to string
    return ' '.join(words)

# Apply cleaning
df['Cleaned_Message'] = df['Message'].apply(clean_text)

# ==============================
# 4️⃣ Feature Extraction
# ==============================
vectorizer = CountVectorizer()
X_vectors = vectorizer.fit_transform(df['Cleaned_Message'])

print("\nShape of feature matrix:", X_vectors.shape)
print("First 20 features:", vectorizer.get_feature_names_out()[:20])

# Features (X) and Target (y)
X = X_vectors
y = df['Category'].str.lower()  # ensure 'spam' and 'ham' are lowercase

# ==============================
# 5️⃣ Train-Test Split & Handling Imbalance
# ==============================
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y  # preserves Spam/Ham ratio
)

print("\nX_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)
print("y_train distribution:\n", y_train.value_counts(normalize=True))
print("y_test distribution:\n", y_test.value_counts(normalize=True))

# ==============================
# 6️⃣ Train Decision Tree
# ==============================
dt_model = DecisionTreeClassifier(
    criterion='gini',        # or 'entropy'
    random_state=42,
    class_weight='balanced'  # handle imbalance
)

dt_model.fit(X_train, y_train)

# ==============================
# 7️⃣ Predictions
# ==============================
y_pred = dt_model.predict(X_test)

# ==============================
# 8️⃣ Evaluation
# ==============================
accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred, pos_label='spam')
recall = recall_score(y_test, y_pred, pos_label='spam')
f1 = f1_score(y_test, y_pred, pos_label='spam')
cm = confusion_matrix(y_test, y_pred, labels=['spam','ham'])

print("\nModel Performance Metrics:")
print("Accuracy:", accuracy)
print("Precision (Spam):", precision)
print("Recall (Spam):", recall)
print("F1-Score (Spam):", f1)
print("Confusion Matrix:\n", cm)
# ==============================
# User Input for Prediction
# ==============================

print("\n--- Predict if Email is Spam or Ham ---")
new_email = input("Enter the email text: ")

# 1️⃣ Clean the email using the same function as training
cleaned_email = clean_text(new_email)

# 2️⃣ Convert to vector using the fitted CountVectorizer
email_vector = vectorizer.transform([cleaned_email])

# 3️⃣ Predict using trained Decision Tree
prediction = dt_model.predict(email_vector)[0]

print("\nPrediction:", prediction.upper())


[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\HP\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


First 5 rows:
  Category                                            Message
0      ham  Go until jurong point, crazy.. Available only ...
1      ham                      Ok lar... Joking wif u oni...
2     spam  Free entry in 2 a wkly comp to win FA Cup fina...
3      ham  U dun say so early hor... U c already then say...
4      ham  Nah I don't think he goes to usf, he lives aro...

Class distribution:
Category
ham     848
spam    152
Name: count, dtype: int64

Percentage distribution:
Category
ham     84.8
spam    15.2
Name: proportion, dtype: float64

Shape of feature matrix: (1000, 3063)
First 20 features: ['aaooooright' 'aathilove' 'aathiwhere' 'aberdeen' 'ability' 'abiola'
 'able' 'abnormally' 'abt' 'aburo' 'ac' 'acc' 'accenture' 'accept'
 'access' 'accidentally' 'accomodate' 'accomodations' 'accordinglyor'
 'account']

X_train shape: (800, 3063)
X_test shape: (200, 3063)
y_train distribution:
 Category
ham     0.8475
spam    0.1525
Name: proportion, dtype: float64
y_test distrib

Enter the email text:  You have been selected as the lucky winner of $10,000.  Click here now to claim your prize before it expires!



Prediction: HAM
